In [ ]:
et italique# Export brut
raw_path = EXPORTS_DIR / "competitors_raw.csv"
df_raw.to_csv(raw_path, index=False)
print(f"Export brut ÃƒÂ¢Ã¢â‚¬Â Ã¢â‚¬â„¢ {raw_path}")

Export brut ÃƒÂ¢Ã¢â‚¬Â Ã¢â‚¬â„¢ C:\Users\33623\Documents\___Projets\AI\Reseaux d'acteurs\analyses\exports\competitors_raw.csv


## 3. Competitor Text Cleaning

**Goal**: convert raw competitor text lists into clean structured lists.

**Processing**: multi-separator split, invalid value removal, case normalization, self-reference removal.
**Output**: `df_clean` with one competitor list per company.

In [196]:
_SEP = re.compile(r"[,;\n]+")
_INVALID = re.compile(r"^(na|n/a|n\\.a\\.?|none|unknown|tbd|-|n\\.?d\\.?|non renseigne)$", re.IGNORECASE)

def clean_surface(value: str) -> str:
    s = str(value).replace("\u00a0", " ").strip()
    s = re.sub(r"\s+", " ", s)
    s = re.sub(r"\s+([,;])", r"\1", s)
    s = s.strip(" \t\r\n,;")
    return s

def split_competitors(raw: str) -> list[str]:
    text = str(raw or "")
    text = text.replace("\u00a0", " ")
    # Keep NA patterns intact before splitting so they can be filtered cleanly.
    text = re.sub(r"\bN\s*/\s*A\b", "NA", text, flags=re.IGNORECASE)
    tokens = [clean_surface(p) for p in _SEP.split(text)]

    out: list[str] = []
    for token in tokens:
        if not token:
            continue
        token_l = token.lower()
        if _INVALID.match(token_l):
            continue
        if token_l in {"n", "a"}:
            continue
        out.append(token)
    return out

df_clean = df_raw.copy()
df_clean[COL_COMPETITORS] = df_clean[COL_COMPETITORS].apply(split_competitors)

df_clean[COL_COMPETITORS] = df_clean.apply(
    lambda r: [c for c in r[COL_COMPETITORS] if c.lower() != str(r[COL_NAME]).lower()],
    axis=1,
 )

df_clean = df_clean[df_clean[COL_COMPETITORS].map(len) > 0].reset_index(drop=True)

print(f"{len(df_clean)} entreprises apres nettoyage texte (sans perte de contexte)")
df_clean[[COL_NAME, COL_COMPETITORS]].head(8)

130 entreprises apres nettoyage texte (sans perte de contexte)


,name,main_competitors
0,Nvidia,"[AMD, Intel, Google, Broadcom, Qualcomm]"
1,Google,"[Microsoft, OpenAI, Perpexity AI, Amazon, byte..."
2,Alphabet,"[Microsoft, Amazon, Apple, Meta, Oracle, OpenA..."
3,Apple,"[Google, Microsoft, Samsung, Amazon, Xiaomi, H..."
4,Microsoft,"[Amazon, Google, Meta, Apple, Sony, Oracle]"
5,Amazon,"[Walmart, Temu, Shein, Alibaba, Microsoft, Goo..."
6,Broadcom,"[Nvidia, Qualcomm, Intel, Marvell Technology]"
7,Meta,"[Google, ByteDance, Snap, X, Apple, SnapChat, ..."


## 4. Semantic Name Normalization

**Goal**: merge subsidiaries, brands, and variants under canonical names.

**Inputs**: `df_clean` and `SEMANTIC_ALIASES`.
**Output**: semantically normalized `df_clean`, deduplicated, without self-competition.

In [197]:
def _norm_key(value: str) -> str:
    return re.sub(r"\s+", " ", str(value).strip()).casefold()

def _strip_parenthetical(value: str) -> str:
    return re.sub(r"\(.*?\)", "", str(value)).strip()

def _inside_parenthetical(value: str) -> str:
    m = re.search(r"\((.*?)\)", str(value))
    return m.group(1).strip() if m else ""

# Canonical names from database.
with sqlite3.connect(DB_PATH) as con:
    df_all_names = pd.read_sql_query("SELECT name FROM enterprises WHERE name IS NOT NULL", con)

canonical_by_key: dict[str, str] = {}
for name in df_all_names["name"].tolist():
    clean = str(name).strip()
    if clean and _norm_key(clean) not in canonical_by_key:
        canonical_by_key[_norm_key(clean)] = clean

# Conventions-driven base aliases.
SEMANTIC_ALIASES: dict[str, str] = {
    # Alphabet / Google ecosystem
    "Alphabet": "Google",
    "Alphabet Inc": "Google",
    "Alphabet Inc.": "Google",
    "YouTube": "Google",
    "Youtube": "Google",
    "Google Search": "Google",
    "Chrome": "Google",
    "Google Translate": "Google",
    "Google Assistant": "Google",
    "Gmail": "Google",
    "Android": "Google",
    "Google Cloud Platform": "Google Cloud",
    "GCP": "Google Cloud",
    "TPU": "Google Cloud",
    "Waymo": "Waymo",
    "DeepMind": "DeepMind",
    "Deepmind": "DeepMind",
    "Google DeepMind": "DeepMind",
    "Google Deepmind": "DeepMind",
    "Antigravity": "DeepMind",
    # Meta
    "Meta Platforms": "Meta",
    "Facebook": "Meta",
    "Instagram": "Meta",
    "WhatsApp": "Meta",
    "Whatsapp": "Meta",
    "Threads": "Meta",
    "Oculus": "Meta",
    "Meta Quest": "Meta",
    "LLaMA": "Meta",
    "Llama": "Meta",
    # Microsoft
    "Microsoft Azure": "Microsoft",
    "Azure": "Microsoft",
    "LinkedIn": "Microsoft",
    "Linkedin": "Microsoft",
    "GitHub": "Microsoft",
    "Github": "Microsoft",
    "Skype": "Microsoft",
    "Bing": "Microsoft",
    "Nuance": "Microsoft",
    "Nuance Communications": "Microsoft",
    "Activision": "Microsoft",
    "Activision Blizzard": "Microsoft",
    "Xbox": "Microsoft",
    "Microsoft Translator": "Microsoft",
    "Office 365": "Microsoft",
    # Quality and frequent variants
    "Openai": "OpenAI",
    "OpenIA": "OpenAI",
    "Deepseek": "DeepSeek",
    "Perpexity Ai": "Perplexity AI",
    "Aws": "AWS",
    "Amazon Web Services": "AWS",
    "Amazon Web Services (AWS)": "AWS",
    "Microsoft(Azure)": "Microsoft",
    "AWS(AWS)": "AWS",
}

ALIASES_BY_KEY = {_norm_key(k): v for k, v in SEMANTIC_ALIASES.items()}

GOOGLE_TO_GOOGLE = {"google", "google search", "youtube", "chrome", "gmail", "android", "google translate", "google assistant"}
GOOGLE_TO_CLOUD_HINTS = {"cloud", "gcp", "tpu", "bigquery", "compute engine", "contact center"}
GOOGLE_TO_DEEPMIND_HINTS = {"deepmind", "antigravity", "gemini"}
GOOGLE_TO_WAYMO_HINTS = {"waymo"}

AI_PEERS = {
    "openai", "anthropic", "deepseek", "mistral", "cohere", "poolside", "hugging face",
    "meta", "black forest labs", "aleph alpha", "helsing", "zhipu ai", "deepmind"
}
CLOUD_PEERS = {
    "aws", "azure", "microsoft azure", "oracle", "alibaba", "ibm", "databricks",
    "snowflake", "redshift", "coreweave", "lambda labs", "google cloud", "google bigquery",
    "google compute engine"
}

def _infer_google_entity(token: str, peers: list[str]) -> tuple[str, str]:
    token_clean = clean_surface(token)
    base = _strip_parenthetical(token_clean).strip()
    inside = _inside_parenthetical(token_clean).strip()
    merged = f"{base} {inside}".strip().casefold()
    base_low = base.casefold()

    if base_low.startswith("alphabet") or base_low in {"alphabet", "alphabet inc", "alphabet inc."}:
        return "Google", "alphabet_to_google"

    if any(h in merged for h in GOOGLE_TO_WAYMO_HINTS):
        return "Waymo", "google_waymo_hint"
    if any(h in merged for h in GOOGLE_TO_CLOUD_HINTS):
        return "Google Cloud", "google_cloud_hint"
    if any(h in merged for h in GOOGLE_TO_DEEPMIND_HINTS):
        return "DeepMind", "google_deepmind_hint"

    if base_low in GOOGLE_TO_GOOGLE:
        return "Google", "google_core_product"

    if base_low == "google":
        peer_text = " | ".join(_norm_key(p) for p in peers)
        has_ai = any(k in peer_text for k in AI_PEERS)
        has_cloud = any(k in peer_text for k in CLOUD_PEERS)
        if has_ai and not has_cloud:
            return "DeepMind", "google_context_ai"
        if has_cloud and not has_ai:
            return "Google Cloud", "google_context_cloud"
        return "Google", "google_default_ambiguous"

    return "Google", "google_fallback"

def _normalize_token(token: str, peers: list[str]) -> tuple[str | None, str]:
    raw = clean_surface(token)
    if not raw:
        return None, "empty"

    raw_low = raw.casefold()
    if _INVALID.match(raw_low) or raw_low in {"n", "a"}:
        return None, "invalid"

    if "google" in raw_low or "alphabet" in raw_low:
        return _infer_google_entity(raw, peers)

    base = _strip_parenthetical(raw)
    base_key = _norm_key(base)
    raw_key = _norm_key(raw)

    if raw_key in ALIASES_BY_KEY:
        return ALIASES_BY_KEY[raw_key], "alias_raw"
    if base_key in ALIASES_BY_KEY:
        return ALIASES_BY_KEY[base_key], "alias_base"
    if raw_key in canonical_by_key:
        return canonical_by_key[raw_key], "canonical_raw"
    if base_key in canonical_by_key:
        return canonical_by_key[base_key], "canonical_base"

    cleaned = re.sub(r"\s+", " ", base).strip()
    return cleaned, "fallback_clean"

def _dedupe_keep_order(values: list[str]) -> list[str]:
    seen: set[str] = set()
    out: list[str] = []
    for v in values:
        k = _norm_key(v)
        if k not in seen:
            seen.add(k)
            out.append(v)
    return out

audit_rows: list[dict[str, str]] = []

def normalize_competitor_list(row: pd.Series) -> list[str]:
    owner = str(row[COL_NAME]).strip()
    items = list(row[COL_COMPETITORS])
    normalized: list[str] = []

    for i, token in enumerate(items):
        peers = items[:i] + items[i + 1 :]
        mapped, reason = _normalize_token(token, peers)
        if mapped is None:
            continue
        if _norm_key(mapped) == _norm_key(owner):
            continue
        normalized.append(mapped)
        if _norm_key(mapped) != _norm_key(token):
            audit_rows.append({
                "company": owner,
                "source": str(token),
                "normalized": mapped,
                "reason": reason,
            })

    return _dedupe_keep_order(normalized)

df_clean[COL_COMPETITORS] = df_clean.apply(normalize_competitor_list, axis=1)
df_clean = df_clean[df_clean[COL_COMPETITORS].map(len) > 0].reset_index(drop=True)

preview = df_clean[[COL_NAME, COL_COMPETITORS]].explode(COL_COMPETITORS)
print(f"{len(SEMANTIC_ALIASES)} aliases de base | {len(df_clean)} entreprises apres normalisation complete")
print("\nTop concurrents apres normalisation:")
print(preview[COL_COMPETITORS].value_counts().head(20).to_string())

df_norm_audit = pd.DataFrame(audit_rows)
norm_audit_path = EXPORTS_DIR / "competitor_normalization_audit.csv"
if not df_norm_audit.empty:
    df_norm_audit.to_csv(norm_audit_path, index=False)
    print(f"\nAudit normalisation -> {norm_audit_path} ({len(df_norm_audit)} remplacements)")
else:
    print("\nAudit normalisation: aucun remplacement trace")

54 aliases de base | 130 entreprises apres normalisation complete

Top concurrents apres normalisation:
main_competitors
Microsoft       29
OpenAI          26
Anthropic       15
DeepMind        14
Meta            13
Amazon          12
Nvidia          11
Google Cloud    11
Google          10
Tesla            9
Apple            8
Baidu            8
SAP              7
Intel            6
Qualcomm         6
ByteDance        6
AWS              6
Waymo            6
Tencent          6
DeepSeek         6

Audit normalisation -> C:\Users\33623\Documents\___Projets\AI\Reseaux d'acteurs\analyses\exports\competitor_normalization_audit.csv (91 remplacements)


## 5. Long Directed Format `company -> competitor`

**Goal**: convert list format into row-level relational format.

**Processing**: explode, canonicalization from database names, directed pair deduplication.
**Output**: `df_long` and `competitors_long.csv`.

In [198]:
def _norm_key(value: str) -> str:
    return re.sub(r"\s+", " ", str(value).strip()).casefold()

# Reference canonicalization from all enterprise names in DB.
with sqlite3.connect(DB_PATH) as con:
    df_all_names = pd.read_sql_query("SELECT name FROM enterprises WHERE name IS NOT NULL", con)

canonical_by_key = {}
for name in df_all_names["name"].tolist():
    clean = str(name).strip()
    if clean:
        key = _norm_key(clean)
        if key not in canonical_by_key:
            canonical_by_key[key] = clean

df_long = (
    df_clean
    .explode(COL_COMPETITORS)
    .rename(columns={COL_COMPETITORS: "competitor"})
    .reset_index(drop=True)
    [[COL_NAME, "competitor", COL_SECTOR, "ranking_score"]]
 )

# Canonicalize competitors to avoid historical case/spelling drift.
df_long["competitor"] = df_long["competitor"].map(lambda c: canonical_by_key.get(_norm_key(c), c))

preferred_canonical = {
    "intel": "Intel",
    "amd": "AMD",
    "bytedanse": "ByteDance",
    "perpexity ai": "Perplexity AI",
    "amazon web services": "AWS",
}
df_long["competitor"] = df_long["competitor"].map(
    lambda c: preferred_canonical.get(_norm_key(c), c)
 )

# Enforce no self-competition after canonicalization.
df_long = df_long[df_long.apply(lambda r: _norm_key(r[COL_NAME]) != _norm_key(r["competitor"]), axis=1)]

# Deduplicate directed company -> competitor pairs.
df_long = (
    df_long.drop_duplicates(subset=[COL_NAME, "competitor"])
    .reset_index(drop=True)
 )
df_long.insert(0, "pair_id", df_long.index)

long_path = EXPORTS_DIR / "competitors_long.csv"
df_long.to_csv(long_path, index=False)

print(f"{len(df_long)} couples entreprise->concurrent apres normalisation et dedoublonnage")
print(f"Entreprises (lignes): {df_long[COL_NAME].nunique()} | Competiteurs (colonnes potentielles): {df_long['competitor'].nunique()}")
print(f"Export -> {long_path}")
df_long.head(10)

817 couples entreprise->concurrent apres normalisation et dedoublonnage
Entreprises (lignes): 130 | Competiteurs (colonnes potentielles): 491
Export -> C:\Users\33623\Documents\___Projets\AI\Reseaux d'acteurs\analyses\exports\competitors_long.csv


,pair_id,name,competitor,sector,ranking_score
0,0,Nvidia,AMD,"Hardware, AI model, ICT",5000000.0
1,1,Nvidia,Intel,"Hardware, AI model, ICT",5000000.0
2,2,Nvidia,Google,"Hardware, AI model, ICT",5000000.0
3,3,Nvidia,Broadcom,"Hardware, AI model, ICT",5000000.0
4,4,Nvidia,Qualcomm,"Hardware, AI model, ICT",5000000.0
5,5,Google,Microsoft,"AI model, Media & Entertainment, Sales & Marke...",4560000.0
6,6,Google,OpenAI,"AI model, Media & Entertainment, Sales & Marke...",4560000.0
7,7,Google,Perplexity AI,"AI model, Media & Entertainment, Sales & Marke...",4560000.0
8,8,Google,Amazon,"AI model, Media & Entertainment, Sales & Marke...",4560000.0
9,9,Google,ByteDance,"AI model, Media & Entertainment, Sales & Marke...",4560000.0


## 6. Relation Aggregation

**Goal**: count frequency for each `company -> competitor` pair.

**Output**: `df_agg` (`name`, `competitor`, `count`) and `competitors_aggregated.csv`.

In [199]:
df_agg = (
    df_long
    .groupby([COL_NAME, "competitor"], sort=False)
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
    .reset_index(drop=True)
)

print("Top 20 paires entrepriseÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Å“concurrent :")
display(df_agg.head(20))

print("\nDistribution des frÃƒÆ’Ã‚Â©quences :")
display(df_agg["count"].describe())

agg_path = EXPORTS_DIR / "competitors_aggregated.csv"
df_agg.to_csv(agg_path, index=False)
print(f"\nExport ÃƒÂ¢Ã¢â‚¬Â Ã¢â‚¬â„¢ {agg_path}")

Top 20 paires entrepriseÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Å“concurrent :


,name,competitor,count
0,Nvidia,AMD,1
1,Nvidia,Intel,1
2,Nvidia,Google,1
3,Nvidia,Broadcom,1
4,Nvidia,Qualcomm,1
5,Google,Microsoft,1
6,Google,OpenAI,1
7,Google,Perplexity AI,1
8,Google,Amazon,1
9,Google,ByteDance,1



Distribution des frÃƒÆ’Ã‚Â©quences :


count    817.0
mean       1.0
std        0.0
min        1.0
25%        1.0
50%        1.0
75%        1.0
max        1.0
Name: count, dtype: float64


Export ÃƒÂ¢Ã¢â‚¬Â Ã¢â‚¬â„¢ C:\Users\33623\Documents\___Projets\AI\Reseaux d'acteurs\analyses\exports\competitors_aggregated.csv


## 7. Directed Matrix `company x competitors`

**Goal**: prepare the matrix representation used by projection and community detection.

**Sample Selection Policy**

We build the analysis sample with explicit, deterministic rules to keep the competition graph interpretable and comparable across runs.

1. **Starting pool**: all companies from the cleaned directed dataset (`df_agg` / `df_raw`).
2. **Investor exclusion**: remove entities identified as investors (e.g., investor/investment, venture capital, VC, private equity).
3. **Relational consistency filter**: remove companies that do not share any competitor with at least one other company.
4. **Rows kept**: only companies passing rules 2 and 3 are kept as matrix rows (analyzed actors).
5. **Columns kept**: competitor columns remain the full mentioned competitor universe from the directed data.

**Why this policy**

- Focus the map on operating competitive dynamics rather than financing actors.
- Remove isolated profiles that cannot contribute to shared competitive structure.
- Preserve a directed representation (`company -> competitor`) while keeping enough overlap for robust community detection.

**Output**: `df_matrix` and `cooccurrence_matrix.csv` (directed matrix).

In [200]:
# Matrice dirigee: lignes = entreprises, colonnes = competiteurs.
# On garde la logique asymetrique, sans M + M^T.
df_matrix = df_agg.pivot_table(
    index=COL_NAME,
    columns="competitor",
    values="count",
    fill_value=0,
)

# Exclude investor-type actors from the selected row set.
sector_series = df_raw[COL_SECTOR].fillna("").astype(str).str.casefold()
investor_mask = sector_series.str.contains(
    r"\binvest(?:or|ment)?s?\b|venture\s+capital|\bvc\b|private\s+equity",
    regex=True,
    na=False,
 )

selected_actor_series = df_raw.loc[~investor_mask, COL_NAME].dropna().astype(str).str.strip()
enterprise_actors = sorted(selected_actor_series[selected_actor_series.ne("")].unique().tolist())

excluded_investors = sorted(
    df_raw.loc[investor_mask, COL_NAME].dropna().astype(str).str.strip().unique().tolist()
 )

df_matrix = df_matrix.reindex(index=enterprise_actors, fill_value=0)

# Keep only companies that share at least one competitor with another company.
binary = (df_matrix > 0).astype(np.int8)
shared_comp_counts = binary.to_numpy(dtype=np.int8) @ binary.to_numpy(dtype=np.int8).T
np.fill_diagonal(shared_comp_counts, 0)
has_shared_competitor = shared_comp_counts.max(axis=1) > 0
isolated_no_common = df_matrix.index[~has_shared_competitor].tolist()
df_matrix = df_matrix.loc[has_shared_competitor].copy()

print(f"Investors excluded from row selection: {len(excluded_investors)}")
if excluded_investors:
    print("Sample excluded investors:", ", ".join(excluded_investors[:12]))

print(f"Companies removed (no shared competitor): {len(isolated_no_common)}")
if isolated_no_common:
    print("Sample removed isolated companies:", ", ".join(isolated_no_common[:12]))

print(f"Matrice dirigee: {df_matrix.shape[0]} entreprises x {df_matrix.shape[1]} competiteurs")
print(f"Densite non-nulle : {(df_matrix.values > 0).mean():.1%}")

cooc_path = EXPORTS_DIR / "cooccurrence_matrix.csv"
df_matrix.to_csv(cooc_path)
print(f"Export matrice dirigee -> {cooc_path}")

# Vulnerability = somme ligne ; Clout = somme colonne.
row_sums = df_matrix.sum(axis=1).rename("vulnerability")
col_sums = df_matrix.sum(axis=0).rename("clout")

row_sums_path = EXPORTS_DIR / "cooccurrence_row_sums.csv"
row_sums.sort_values(ascending=False).to_frame().to_csv(row_sums_path)
print(f"Export vulnerability (row sums) -> {row_sums_path}")

col_sums_path = EXPORTS_DIR / "cooccurrence_column_sums.csv"
col_sums.sort_values(ascending=False).to_frame().to_csv(col_sums_path)
print(f"Export clout (column sums) -> {col_sums_path}")

df_vulnerability_clout = pd.DataFrame({"actor": df_matrix.index.astype(str)})
df_vulnerability_clout["vulnerability"] = df_vulnerability_clout["actor"].map(row_sums).fillna(0)
df_vulnerability_clout["clout"] = df_vulnerability_clout["actor"].map(col_sums).fillna(0)
cap_num = pd.to_numeric(df_raw["capitalization"], errors="coerce") if "capitalization" in df_raw.columns else pd.Series(np.nan, index=df_raw.index)
funds_num = pd.to_numeric(df_raw["funds_raised"], errors="coerce") if "funds_raised" in df_raw.columns else pd.Series(np.nan, index=df_raw.index)
ranking_num = pd.to_numeric(df_raw["ranking_score"], errors="coerce")
size_basis_series = cap_num.fillna(funds_num).fillna(ranking_num).clip(lower=0)
size_basis_map = pd.Series(size_basis_series.to_numpy(), index=df_raw[COL_NAME]).to_dict()
df_vulnerability_clout["size_basis"] = df_vulnerability_clout["actor"].map(size_basis_map).fillna(0)
df_vulnerability_clout = df_vulnerability_clout.sort_values(["vulnerability", "clout"], ascending=[False, False])
cap_min = float(df_vulnerability_clout["size_basis"].min())
cap_max = float(df_vulnerability_clout["size_basis"].max())
if np.isclose(cap_min, cap_max):
    df_vulnerability_clout["bubble_size"] = 12.0
else:
    df_vulnerability_clout["bubble_size"] = 8 + (df_vulnerability_clout["size_basis"] - cap_min) * (26 - 8) / (cap_max - cap_min)
rng = np.random.default_rng(RANDOM_SEED)
jitter_x = 0.12
jitter_y = 0.12
df_vulnerability_clout["clout_jitter"] = df_vulnerability_clout["clout"] + rng.normal(0, jitter_x, len(df_vulnerability_clout))
df_vulnerability_clout["vulnerability_jitter"] = df_vulnerability_clout["vulnerability"] + rng.normal(0, jitter_y, len(df_vulnerability_clout))
vuln_clout_path = EXPORTS_DIR / "enterprise_vulnerability_clout.csv"
df_vulnerability_clout.to_csv(vuln_clout_path, index=False)
print(f"Export vulnerability+clout -> {vuln_clout_path}")

# Scatterplot clout vs vulnerability.
import plotly.express as px

fig_vuln_clout = px.scatter(
    df_vulnerability_clout,
    x="clout_jitter",
    y="vulnerability_jitter",
    size="bubble_size",
    size_max=26,
    hover_name="actor",
    hover_data={
        "clout": True,
        "vulnerability": True,
        "size_basis": ":,.0f",
        "bubble_size": False,
        "clout_jitter": False,
        "vulnerability_jitter": False,
    },
    title="Clout vs Vulnerability",
    labels={
        "clout_jitter": "Clout (jittered)",
        "vulnerability_jitter": "Vulnerability (jittered)",
    },
)
fig_vuln_clout.update_traces(marker=dict(opacity=0.8, line=dict(width=0.5, color="#1f2937")))
fig_vuln_clout.update_layout(plot_bgcolor="#FDFAF4", paper_bgcolor="#FDFAF4")

vuln_clout_fig_html = EXPORTS_DIR / "vulnerability_clout_scatter.html"
fig_vuln_clout.write_html(str(vuln_clout_fig_html))
print(f"Export clout-vulnerability scatter -> {vuln_clout_fig_html}")

vuln_clout_fig_png = EXPORTS_DIR / "vulnerability_clout_scatter.png"
try:
    fig_vuln_clout.write_image(str(vuln_clout_fig_png), width=1400, height=900, scale=2)
    print(f"PNG export -> {vuln_clout_fig_png}")
except Exception as e:
    print(f"PNG export skipped ({type(e).__name__}): {e}")

fig_vuln_clout.show()

Investors excluded from row selection: 1
Sample excluded investors: SoftBank Group
Companies removed (no shared competitor): 20
Sample removed isolated companies: 4.screen, DarwinAI, DentalMonitoring, Dust, EbenBuild, Franka Emika, Gleamer, Harvey, Job&Talent, Langfuse, ManoMano, Microchip Technology Inc
Matrice dirigee: 111 entreprises x 491 competiteurs
Densite non-nulle : 1.3%
Export matrice dirigee -> C:\Users\33623\Documents\___Projets\AI\Reseaux d'acteurs\analyses\exports\cooccurrence_matrix.csv
Export vulnerability (row sums) -> C:\Users\33623\Documents\___Projets\AI\Reseaux d'acteurs\analyses\exports\cooccurrence_row_sums.csv
Export clout (column sums) -> C:\Users\33623\Documents\___Projets\AI\Reseaux d'acteurs\analyses\exports\cooccurrence_column_sums.csv
Export vulnerability+clout -> C:\Users\33623\Documents\___Projets\AI\Reseaux d'acteurs\analyses\exports\enterprise_vulnerability_clout.csv
Export clout-vulnerability scatter -> C:\Users\33623\Documents\___Projets\AI\Reseaux d

## 7.b Selection Audit

**Goal**: document and monitor sample-selection effects at each run.

This audit reports:
- total candidate companies before filtering,
- exclusions due to investor profile,
- exclusions due to no shared competitor,
- final retained companies in the analysis matrix.

In [201]:
# Selection audit report (derived from matrix-filter variables).
base_candidates = sorted(df_raw[COL_NAME].dropna().astype(str).str.strip().unique().tolist())
base_candidates = [n for n in base_candidates if n]

excluded_investors_set = set(excluded_investors) if "excluded_investors" in globals() else set()
excluded_isolated_set = set(isolated_no_common) if "isolated_no_common" in globals() else set()
kept_set = set(df_matrix.index.astype(str).tolist())

audit_summary = pd.DataFrame(
    [
        {"stage": "base_candidates", "count": len(base_candidates)},
        {"stage": "excluded_investors", "count": len(excluded_investors_set)},
        {"stage": "excluded_no_shared_competitor", "count": len(excluded_isolated_set)},
        {"stage": "kept_in_matrix", "count": len(kept_set)},
    ]
)

status_rows = []
for name in base_candidates:
    is_investor = name in excluded_investors_set
    is_isolated = name in excluded_isolated_set
    kept = name in kept_set
    if kept:
        status = "kept"
    elif is_investor and is_isolated:
        status = "excluded_investor_and_isolated"
    elif is_investor:
        status = "excluded_investor"
    elif is_isolated:
        status = "excluded_no_shared_competitor"
    else:
        status = "not_in_matrix_other_reason"
    status_rows.append({
        "actor": name,
        "status": status,
        "is_investor": bool(is_investor),
        "is_isolated_no_shared_competitor": bool(is_isolated),
        "kept_in_matrix": bool(kept),
    })

audit_details = pd.DataFrame(status_rows).sort_values(["status", "actor"]).reset_index(drop=True)

summary_path = EXPORTS_DIR / "selection_audit_summary.csv"
details_path = EXPORTS_DIR / "selection_audit_details.csv"
audit_summary.to_csv(summary_path, index=False)
audit_details.to_csv(details_path, index=False)

print("Selection audit summary:")
display(audit_summary)

print("\nExcluded investors (first 20):")
display(audit_details[audit_details["status"] == "excluded_investor"].head(20))

print("\nExcluded for no shared competitor (first 20):")
display(audit_details[audit_details["status"] == "excluded_no_shared_competitor"].head(20))

print(f"\nExport summary -> {summary_path}")
print(f"Export details -> {details_path}")

Selection audit summary:


,stage,count
0,base_candidates,132
1,excluded_investors,1
2,excluded_no_shared_competitor,20
3,kept_in_matrix,111



Excluded investors (first 20):


,actor,status,is_investor,is_isolated_no_shared_competitor,kept_in_matrix
0,SoftBank Group,excluded_investor,True,False,False



Excluded for no shared competitor (first 20):


,actor,status,is_investor,is_isolated_no_shared_competitor,kept_in_matrix
1,4.screen,excluded_no_shared_competitor,False,True,False
2,DarwinAI,excluded_no_shared_competitor,False,True,False
3,DentalMonitoring,excluded_no_shared_competitor,False,True,False
4,Dust,excluded_no_shared_competitor,False,True,False
5,EbenBuild,excluded_no_shared_competitor,False,True,False
6,Franka Emika,excluded_no_shared_competitor,False,True,False
7,Gleamer,excluded_no_shared_competitor,False,True,False
8,Harvey,excluded_no_shared_competitor,False,True,False
9,Job&Talent,excluded_no_shared_competitor,False,True,False
10,Langfuse,excluded_no_shared_competitor,False,True,False



Export summary -> C:\Users\33623\Documents\___Projets\AI\Reseaux d'acteurs\analyses\exports\selection_audit_summary.csv
Export details -> C:\Users\33623\Documents\___Projets\AI\Reseaux d'acteurs\analyses\exports\selection_audit_details.csv


cela cohÃ©rence du script## 8. 2D Projection (t-SNE)

**Goal**: project company competition profiles directly in 2D for all downstream analyses and visualizations.

**Input**: L2-normalized `df_matrix`.
**Key parameter**: `perplexity = 10`.
**Output**: `df_coords` with (`x`, `y`) and export `coords_2d.csv`.

In [202]:
N = df_matrix.shape[0]
X = normalize(df_matrix.values, norm="l2")

perplexity = 10
tsne_dims = 2
print(f"t-SNE {tsne_dims}D on {N} companies (perplexity={perplexity})")

coords = TSNE(
    n_components=tsne_dims,
    perplexity=perplexity,
    init="pca",
    learning_rate="auto",
    metric="cosine",
    random_state=RANDOM_SEED,
).fit_transform(X)

method = "t-SNE-2D"

out_degree = df_matrix.sum(axis=1).to_dict()
ranking_score_map = df_raw.set_index(COL_NAME)["ranking_score"].to_dict()

enterprise_names = list(df_matrix.index)
df_coords = pd.DataFrame({
    "actor": enterprise_names,
    "x": coords[:, 0],
    "y": coords[:, 1],
})

df_coords["score"] = df_coords["actor"].map(out_degree).fillna(0)
df_coords["log_score"] = np.log10(df_coords["score"] + 1)
df_coords["ranking_score"] = df_coords["actor"].map(ranking_score_map).fillna(0)

sector_map = (
    df_raw.set_index(COL_NAME)[COL_SECTOR]
    .dropna()
    .apply(lambda s: s.split(",")[0].strip())
    .to_dict()
)
df_coords["sector"] = df_coords["actor"].map(sector_map).fillna("Unknown")

coords_path = EXPORTS_DIR / "coords_2d.csv"
df_coords.to_csv(coords_path, index=False)
print(f"2D coordinates ({method}) -> {coords_path}")
df_coords.sort_values("ranking_score", ascending=False).head(10)

t-SNE 2D on 111 companies (perplexity=10)
2D coordinates (t-SNE-2D) -> C:\Users\33623\Documents\___Projets\AI\Reseaux d'acteurs\analyses\exports\coords_2d.csv


,actor,x,y,score,log_score,ranking_score,sector
72,Nvidia,-13.441331,2.547343,5.0,0.778151,5000000.0,Hardware
50,Google,-0.968354,4.199326,8.0,0.954243,4560000.0,AI model
11,Alphabet,0.473929,3.677158,10.0,1.041393,4120000.0,Cloud Provider
17,Apple,-2.528941,1.302136,17.0,1.255273,4000000.0,ICT
64,Microsoft,1.420701,1.236174,6.0,0.845098,3450000.0,Cloud Provider
12,Amazon,-0.263396,-1.401854,10.0,1.041393,2900000.0,Unknown
22,Broadcom,-17.136343,1.424355,4.0,0.698970,1835000.0,Hardware
63,Meta,2.915629,3.122268,12.0,1.113943,1503000.0,AI model
96,SpaceX,-7.138571,-17.808071,8.0,0.954243,1480000.0,Cloud Provider
6,AWS,-3.072612,-16.622129,5.0,0.778151,1200000.0,Hardware


## 9. Final Community Map (Louvain on 2D Coordinates)

**Goal**: produce the final map with graph-based communities, fully computed in 2D.

**Method**:
- Build a k-nearest-neighbors graph from 2D coordinates (`x`, `y`).
- Weight edges by distance similarity.
- Run Louvain community detection on this graph.
- Render labels and blobs in the same 2D space.

**Community naming policy**:
- Labels describe shared activities inferred from company descriptions and sectors.
- Company names are never used as community labels.
- Recognizable ecosystems receive concise names (for example, `GAFAM`).
- Otherwise, labels combine a dominant data-grounded theme with a distinctive vocabulary.

**Visual rules**:
- Label color by country (custom country palette).
- Label size from `ranking_score` in `sqrt` mode (range 8-22 after cap).
- Blob color uses an extended discrete palette by community.
- Blob opacity is low but visible (`0.13`).
- Labels only (no point markers).

**Outputs**:
- `communities_kmeans_2d.csv` (community assignments)
- `community_labels_short.csv` (data-grounded community labels)
- `competition_map_2d_kmeans.html` (interactive 2D final map)

In [203]:
import plotly.graph_objects as go

info_cols = [
    "founded_year", "employees_count", "revenue_millions",
    "capitalization", "funds_raised", "description"
]

# Strategie de taille des labels (solution recommandee)
SIZE_MODE = "sqrt"  # options: linear, linear_clip_p99, log10, log10_clip_p99, log1p, log1p_clip_p99, sqrt, sqrt_clip_p99
SIZE_MIN, SIZE_MAX = 8, 18
SIZE_FALLBACK = 10
SIZE_CLIP_Q = 0.99


def country_key(value: str) -> str:
    return str(value).strip().casefold()


def scale_to_range(values: pd.Series, min_size: float, max_size: float, fallback_size: float) -> pd.Series:
    if values.isna().all() or np.isclose(values.max(), values.min()):
        return pd.Series(fallback_size, index=values.index)
    scaled = min_size + (values - values.min()) * (max_size - min_size) / (values.max() - values.min())
    return scaled


def compute_label_sizes(
    raw_scores: pd.Series,
    mode: str,
    min_size: float,
    max_size: float,
    fallback_size: float,
    clip_q: float,
    ) -> tuple[pd.Series, dict]:
    scores = pd.to_numeric(raw_scores, errors="coerce").where(lambda s: s > 0)
    if not scores.notna().any():
        return pd.Series(fallback_size, index=raw_scores.index), {"mode": mode, "clip_value": None}

    use_clip = "_clip_p99" in mode
    base_mode = mode.replace("_clip_p99", "")

    working = scores.copy()
    clip_value = None
    if use_clip:
        clip_value = float(working.quantile(clip_q))
        working = working.clip(upper=clip_value)

    if base_mode == "linear":
        transformed = working
    elif base_mode == "log10":
        transformed = np.log10(working)
    elif base_mode == "log1p":
        transformed = np.log1p(working)
    elif base_mode == "sqrt":
        transformed = np.sqrt(working)
    else:
        raise ValueError(f"SIZE_MODE inconnu: {mode}")

    sizes = scale_to_range(transformed, min_size=min_size, max_size=max_size, fallback_size=fallback_size).round(1)
    return sizes, {"mode": mode, "clip_value": clip_value}


def apply_label_repel(
    df: pd.DataFrame,
    x_col: str = "x",
    y_col: str = "y",
    size_col: str = "label_size",
    iterations: int = 320,
    anchor_pull: float = 0.02,
    step: float = 0.55,
    ) -> pd.DataFrame:
    """
    Petit moteur de repulsion (type force layout) pour limiter les chevauchements de labels.
    La simulation tourne en coordonnees normalisees [0,1] puis revient en coordonnees d'origine.
    """
    x = pd.to_numeric(df[x_col], errors="coerce").to_numpy(dtype=float)
    y = pd.to_numeric(df[y_col], errors="coerce").to_numpy(dtype=float)
    s = pd.to_numeric(df[size_col], errors="coerce").fillna(SIZE_FALLBACK).to_numpy(dtype=float)

    n = len(df)
    if n <= 1:
        return pd.DataFrame({"x_label": x, "y_label": y}, index=df.index)

    xmin, xmax = float(np.min(x)), float(np.max(x))
    ymin, ymax = float(np.min(y)), float(np.max(y))
    xspan = xmax - xmin if not np.isclose(xmax, xmin) else 1.0
    yspan = ymax - ymin if not np.isclose(ymax, ymin) else 1.0

    pos = np.column_stack([(x - xmin) / xspan, (y - ymin) / yspan])
    origin = pos.copy()

    smin, smax = float(np.min(s)), float(np.max(s))
    sspan = smax - smin if not np.isclose(smax, smin) else 1.0
    radii = 0.010 + ((s - smin) / sspan) * 0.028

    for _ in range(iterations):
        disp = np.zeros_like(pos)

        for i in range(n - 1):
            delta = pos[i] - pos[i + 1 :]
            dist = np.linalg.norm(delta, axis=1) + 1e-9
            target = radii[i] + radii[i + 1 :]
            overlap = target - dist
            mask = overlap > 0

            if np.any(mask):
                force = (overlap[mask] / dist[mask])[:, None] * delta[mask]
                move = force * step
                disp[i] += move.sum(axis=0)
                disp[i + 1 :][mask] -= move

        disp += (origin - pos) * anchor_pull
        pos += disp
        pos = np.clip(pos, 0.0, 1.0)

    x_label = pos[:, 0] * xspan + xmin
    y_label = pos[:, 1] * yspan + ymin
    return pd.DataFrame({"x_label": x_label, "y_label": y_label}, index=df.index)


def build_country_colors(countries: list[str]) -> dict[str, str]:
    forced_colors = {
        "china": "#D62828",                     # rouge
        "united states": "#1D4ED8",             # bleu
        "united states of america": "#1D4ED8",  # bleu
        "usa": "#1D4ED8",                       # bleu
        "canada": "#60A5FA",                    # bleu clair
        "japan": "#EC4899",                     # rose
        "south korea": "#EC4899",               # rose
        "korea, south": "#EC4899",              # rose
        "republic of korea": "#EC4899",         # rose
    }

    europe_country_keys = {
        "albania", "andorra", "austria", "belarus", "belgium", "bosnia and herzegovina",
        "bulgaria", "croatia", "cyprus", "czechia", "czech republic", "denmark", "estonia",
        "finland", "france", "germany", "greece", "hungary", "iceland", "ireland", "italy",
        "latvia", "liechtenstein", "lithuania", "luxembourg", "malta", "moldova", "monaco",
        "montenegro", "netherlands", "north macedonia", "norway", "poland", "portugal", "romania",
        "san marino", "serbia", "slovakia", "slovenia", "spain", "sweden", "switzerland",
        "ukraine", "united kingdom", "vatican city", "kosovo"
    }

    greens = [
        "#1B4332", "#24553F", "#2D6A4F", "#3A7D5D", "#4C956C", "#5FAF7D",
        "#74C69D", "#52B788", "#40916C", "#3E8E63", "#2F7D57", "#2A6F54"
    ]

    fallback_palette = [
        "#6D597A", "#E76F51", "#264653", "#457B9D", "#BC6C25", "#B56576",
        "#3A86FF", "#FF006E", "#0A9396", "#7F5539", "#4361EE", "#FF7F11",
        "#2B2D42", "#8D99AE", "#8338EC", "#3D405B"
    ]

    color_map = {}
    europe = sorted([c for c in countries if country_key(c) in europe_country_keys])
    non_europe = [c for c in countries if c not in europe]

    for i, country in enumerate(europe):
        color_map[country] = greens[i % len(greens)]

    fallback_idx = 0
    for country in non_europe:
        key = country_key(country)
        if key in forced_colors:
            color_map[country] = forced_colors[key]
        else:
            color_map[country] = fallback_palette[fallback_idx % len(fallback_palette)]
            fallback_idx += 1

    return color_map


def format_hover(row: pd.Series) -> str:
    lines = [f"<b>{row['actor']}</b>"]

    if pd.notna(row.get("sector")):
        lines.append(f"Sector: {row['sector']}")
    if pd.notna(row.get("country")):
        lines.append(f"Country: {row['country']}")
    if pd.notna(row.get("founded_year")):
        lines.append(f"Founded: {int(row['founded_year'])}")
    if pd.notna(row.get("employees_count")):
        lines.append(f"Employees: {int(row['employees_count']):,}")

    cap = row.get("capitalization_num")
    if pd.notna(cap) and cap > 0:
        lines.append(f"Market cap: {cap/1000:.1f}B USD")

    rev = row.get("revenue_num")
    if pd.notna(rev) and rev > 0:
        lines.append(f"Revenue: {rev/1000:.1f}B USD")

    if pd.notna(row.get("description")):
        desc = str(row["description"])
        snippet = desc[:160].rstrip()
        lines.append(f"<i>{snippet}{'ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¦' if len(desc) > 160 else ''}</i>")

    lines.append(f"Outgoing competitor links: {int(row['score'])}")
    return "<br>".join(lines)


# Assemble les donnees de plotting.
df_plot = df_coords.copy()
available = [c for c in info_cols if c in df_raw.columns]
if available:
    df_info = df_raw.set_index(COL_NAME)[available]
    for col in available:
        df_plot[col] = df_plot["actor"].map(df_info[col])

# Ajoute le pays avec fallback SQL si la colonne manque dans df_raw.
if "country" in df_raw.columns:
    country_map = df_raw.set_index(COL_NAME)["country"]
else:
    with sqlite3.connect(DB_PATH) as con:
        df_country = pd.read_sql_query(f"SELECT name AS {COL_NAME}, country FROM enterprises", con)
    country_map = df_country.set_index(COL_NAME)["country"]

df_plot["country"] = df_plot["actor"].map(country_map).fillna("Unknown")

# Colonnes numeriques utiles pour hover/tailles.
df_plot["capitalization_num"] = pd.to_numeric(df_plot["capitalization"], errors="coerce") if "capitalization" in df_plot.columns else np.nan
df_plot["revenue_num"] = pd.to_numeric(df_plot["revenue_millions"], errors="coerce") if "revenue_millions" in df_plot.columns else np.nan

# Taille des labels: mode configurable, defaut log1p + clipping P99.
df_plot["label_size"], size_meta = compute_label_sizes(
    raw_scores=df_plot["ranking_score"],
    mode=SIZE_MODE,
    min_size=SIZE_MIN,
    max_size=SIZE_MAX,
    fallback_size=SIZE_FALLBACK,
    clip_q=SIZE_CLIP_Q,
    )

# Repositionnement repel en deux passes: global puis micro-ajustement intra-pays.
label_xy = apply_label_repel(
    df_plot,
    x_col="x",
    y_col="y",
    size_col="label_size",
    iterations=380,
    anchor_pull=0.018,
    step=0.62,
    )
df_plot["x_label"] = label_xy["x_label"]
df_plot["y_label"] = label_xy["y_label"]

for country_name, idx in df_plot.groupby("country").groups.items():
    sub = df_plot.loc[idx, ["x_label", "y_label", "label_size"]].rename(
        columns={"x_label": "x", "y_label": "y"}
    )
    sub_xy = apply_label_repel(
        sub,
        x_col="x",
        y_col="y",
        size_col="label_size",
        iterations=160,
        anchor_pull=0.045,
        step=0.36,
    )
    df_plot.loc[idx, "x_label"] = sub_xy["x_label"].to_numpy()
    df_plot.loc[idx, "y_label"] = sub_xy["y_label"].to_numpy()

mean_shift = np.mean(np.sqrt((df_plot["x_label"] - df_plot["x"]) ** 2 + (df_plot["y_label"] - df_plot["y"]) ** 2))
print(f"Repel shift moyen: {mean_shift:.2f} (unitÃƒÆ’Ã‚Â©s de projection)")
print(f"Label size stats -> min: {df_plot['label_size'].min():.1f}, max: {df_plot['label_size'].max():.1f}")
print(f"Label size basis -> {size_meta['mode']} on ranking_score")
if size_meta["clip_value"] is not None:
    print(f"Clipping upper bound (P99): {size_meta['clip_value']:.3g}")

# Hover + couleurs par pays.
df_plot["hover"] = df_plot.apply(format_hover, axis=1)
countries = sorted(df_plot["country"].dropna().unique().tolist())
country_colors = build_country_colors(countries)

fig = go.Figure()
for country in countries:
    sub = df_plot[df_plot["country"] == country].copy().sort_values("label_size", ascending=True)

    sub["label_text"] = sub["actor"].combine(sub["label_size"], lambda actor, size: (
        f"<span style='font-size:{size}px;font-weight:400;"
        "text-shadow:0 0 1px rgba(253,250,244,0.95),0 0 3px rgba(253,250,244,0.65)'>"
        f"{actor}</span>"
    ))

    fig.add_trace(
        go.Scatter(
            x=sub["x_label"],
            y=sub["y_label"],
            mode="text",
            text=sub["label_text"],
            textposition="middle center",
            textfont=dict(color=country_colors[country]),
            name=country,
            customdata=np.stack([sub["hover"]], axis=-1),
            hovertemplate="%{customdata[0]}<extra></extra>",
            showlegend=True,
        )
    )

fig.update_layout(
    title=f"{method} ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â Entreprises dans l'espace des competiteurs ({N} entreprises)",
    xaxis=dict(title="Dimension 1", showgrid=False, zeroline=False),
    yaxis=dict(title="Dimension 2", showgrid=False, zeroline=False),
    legend=dict(title="Pays", font=dict(size=11)),
    font=dict(family="Inter, sans-serif", size=12),
    plot_bgcolor="#FDFAF4",
    paper_bgcolor="#FDFAF4",
    hovermode="closest",
    width=1400,
    height=1200,
    )

fig_html = EXPORTS_DIR / "competition_map_2d.html"
fig.write_html(str(fig_html))
print(f"Carte interactive ({method}) -> {fig_html}")
fig.show()

Repel shift moyen: 0.13 (unitÃƒÆ’Ã‚Â©s de projection)
Label size stats -> min: 8.0, max: 18.0
Label size basis -> sqrt on ranking_score
Carte interactive (t-SNE-2D) -> C:\Users\33623\Documents\___Projets\AI\Reseaux d'acteurs\analyses\exports\competition_map_2d.html


In [204]:
import networkx as nx
from sklearn.neighbors import NearestNeighbors
import plotly.graph_objects as go
import plotly.express as px

if "df_coords" not in globals() or df_coords.empty:
    raise RuntimeError("df_coords is missing. Run Cell 14 first.")

required_cols = ["x", "y"]
missing = [c for c in required_cols if c not in df_coords.columns]
if missing:
    raise RuntimeError(f"df_coords is missing required columns: {missing}")

k_neighbors = 3
louvain_resolution = 3.2
min_refined_size = 1
community_method = f"louvain_2d(k_neighbors={k_neighbors},resolution={louvain_resolution},min_refined_size={min_refined_size})"

FINAL_LABEL_MAX = 22
LABEL_REPEL_ITERS = 130
LABEL_REPEL_ANCHOR = 0.035
LABEL_REPEL_STEP = 0.28
BLOB_FILL_ALPHA = 0.08


def _hex_to_rgba(color: str, alpha: float) -> str:
    s = str(color).strip()

    if s.lower().startswith("rgb") and "(" in s and ")" in s:
        inside = s[s.find("(") + 1 : s.rfind(")")]
        parts = [p.strip() for p in inside.split(",")]
        if len(parts) >= 3:
            try:
                r = int(float(parts[0]))
                g = int(float(parts[1]))
                b = int(float(parts[2]))
                r = max(0, min(255, r))
                g = max(0, min(255, g))
                b = max(0, min(255, b))
                return f"rgba({r},{g},{b},{alpha})"
            except ValueError:
                pass

    h = s.lstrip("#")
    if len(h) == 3:
        h = "".join(ch * 2 for ch in h)
    if len(h) == 6:
        try:
            r = int(h[0:2], 16)
            g = int(h[2:4], 16)
            b = int(h[4:6], 16)
            return f"rgba({r},{g},{b},{alpha})"
        except ValueError:
            pass

    return f"rgba(107,114,128,{alpha})"


def _convex_hull(points: np.ndarray) -> np.ndarray:
    pts = np.asarray(points, dtype=float)
    if len(pts) <= 1:
        return pts
    pts = pts[np.lexsort((pts[:, 1], pts[:, 0]))]

    def _cross(o, a, b):
        return (a[0] - o[0]) * (b[1] - o[1]) - (a[1] - o[1]) * (b[0] - o[0])

    lower = []
    for p in pts:
        while len(lower) >= 2 and _cross(lower[-2], lower[-1], p) <= 0:
            lower.pop()
        lower.append(p)

    upper = []
    for p in pts[::-1]:
        while len(upper) >= 2 and _cross(upper[-2], upper[-1], p) <= 0:
            upper.pop()
        upper.append(p)

    return np.array(lower[:-1] + upper[:-1], dtype=float)


def _expand_polygon(poly: np.ndarray, scale: float = 1.14) -> np.ndarray:
    if poly.shape[0] == 0:
        return poly
    center = poly.mean(axis=0)
    return center + (poly - center) * scale


def _base_blob_polygon(points_xy: np.ndarray, x_range: float, y_range: float) -> np.ndarray:
    n = len(points_xy)
    if n >= 3:
        hull = _convex_hull(points_xy)
        return _expand_polygon(hull, scale=1.14)

    pad = max(0.006 * max(x_range, y_range), 0.35)
    if n == 2:
        p1, p2 = points_xy[0], points_xy[1]
        v = p2 - p1
        nv = np.linalg.norm(v)
        if nv < 1e-12:
            nv = 1.0
            v = np.array([1.0, 0.0])
        v = v / nv
        perp = np.array([-v[1], v[0]])
        p1a = p1 - v * pad
        p2a = p2 + v * pad
        return np.array([p1a + perp * pad, p2a + perp * pad, p2a - perp * pad, p1a - perp * pad], dtype=float)

    c = points_xy[0]
    theta = np.linspace(0, 2 * np.pi, 24, endpoint=False)
    rx = pad
    ry = pad
    return np.column_stack([c[0] + rx * np.cos(theta), c[1] + ry * np.sin(theta)])


def _patatoid_blob(points_xy: np.ndarray, x_range: float, y_range: float, seed: int) -> np.ndarray:
    poly = _base_blob_polygon(points_xy, x_range=x_range, y_range=y_range)
    if poly.shape[0] < 3:
        return poly

    center = poly.mean(axis=0)
    vec = poly - center
    theta = np.arctan2(vec[:, 1], vec[:, 0])
    radius = np.linalg.norm(vec, axis=1)

    order = np.argsort(theta)
    theta_s = theta[order]
    radius_s = radius[order]

    theta_ext = np.r_[theta_s, theta_s[0] + 2 * np.pi]
    radius_ext = np.r_[radius_s, radius_s[0]]

    theta_u = np.linspace(theta_s[0], theta_s[0] + 2 * np.pi, 140, endpoint=False)
    radius_u = np.interp(theta_u, theta_ext, radius_ext)

    rng = np.random.default_rng(seed)
    phase1 = float(rng.uniform(-np.pi, np.pi))
    phase2 = float(rng.uniform(-np.pi, np.pi))
    amp1 = float(rng.uniform(0.07, 0.12))
    amp2 = float(rng.uniform(0.04, 0.08))

    wobble = 1.0 + amp1 * np.sin(3.0 * theta_u + phase1) + amp2 * np.sin(5.0 * theta_u + phase2)
    radius_p = np.maximum(radius_u * wobble, np.percentile(radius_u, 15) * 0.85)

    x = center[0] + radius_p * np.cos(theta_u)
    y = center[1] + radius_p * np.sin(theta_u)
    return np.column_stack([x, y])


def _community_compactness(sub: pd.DataFrame) -> float:
    pts = sub[["x", "y"]].to_numpy(float)
    if len(pts) <= 1:
        return 0.0
    cent = pts.mean(axis=0)
    d = np.sqrt(((pts - cent) ** 2).sum(axis=1))
    return float(np.mean(d))


def _compactness_to_blob_style(c_id: int) -> tuple[str, float]:
    palette = [
        "#E76F51", "#2A9D8F", "#F4A261", "#457B9D", "#E63946",
        "#43AA8B", "#577590", "#F8961E", "#277DA1", "#B56576",
        "#90BE6D", "#4D908E", "#BC6C25", "#6D597A", "#3D405B",
    ]
    color = palette[int(c_id) % len(palette)]
    return color, float(BLOB_FILL_ALPHA)


XY = df_coords[["x", "y"]].to_numpy(dtype=float)
n_points = XY.shape[0]
k_eff = max(2, min(k_neighbors + 1, n_points))

knn = NearestNeighbors(n_neighbors=k_eff, metric="euclidean")
knn.fit(XY)
distances, indices = knn.kneighbors(XY)

sigma = float(np.median(distances[:, 1:])) if np.any(distances[:, 1:] > 0) else 1.0
sigma = max(sigma, 1e-6)

G = nx.Graph()
for i, actor in enumerate(df_coords["actor"].tolist()):
    G.add_node(i, actor=actor)

for i in range(n_points):
    for d, j in zip(distances[i, 1:], indices[i, 1:]):
        if i == j:
            continue
        w = float(np.exp(-(d ** 2) / (2.0 * sigma ** 2)))
        if G.has_edge(i, j):
            if w > G[i][j]["weight"]:
                G[i][j]["weight"] = w
        else:
            G.add_edge(i, j, weight=w)

communities = nx.algorithms.community.louvain_communities(
    G,
    weight="weight",
    resolution=louvain_resolution,
    seed=RANDOM_SEED,
)

community_by_node = {}
for c_id, nodes in enumerate(communities):
    for node in nodes:
        community_by_node[node] = c_id

df_cluster_labels = df_coords.copy()
df_cluster_labels["community"] = [community_by_node[i] for i in range(n_points)]

country_map = {}
if "country" in df_raw.columns:
    country_map = df_raw.set_index(COL_NAME)["country"].to_dict()
else:
    with sqlite3.connect(DB_PATH) as con:
        rows = con.execute("SELECT name, country FROM enterprises").fetchall()
    country_map = {str(n): c for n, c in rows}

sector_map = df_raw.set_index(COL_NAME)[COL_SECTOR].to_dict()
df_cluster_labels["country"] = df_cluster_labels["actor"].map(country_map).fillna("Unknown")
df_cluster_labels["sector"] = (
    df_cluster_labels["actor"].map(sector_map).fillna("Unknown").astype(str).str.split(",").str[0].str.strip()
)

if "df_plot" in globals() and "label_size" in df_plot.columns:
    label_size_map = df_plot.set_index("actor")["label_size"].to_dict()
    df_cluster_labels["label_size"] = df_cluster_labels["actor"].map(label_size_map).fillna(SIZE_FALLBACK)
else:
    df_cluster_labels["label_size"], _ = compute_label_sizes(
        raw_scores=df_cluster_labels["ranking_score"],
        mode=SIZE_MODE,
        min_size=SIZE_MIN,
        max_size=SIZE_MAX,
        fallback_size=SIZE_FALLBACK,
        clip_q=SIZE_CLIP_Q,
    )

label_sizes = pd.to_numeric(df_cluster_labels["label_size"], errors="coerce").fillna(SIZE_FALLBACK)
size_min = float(label_sizes.min())
size_max = float(label_sizes.max())
if np.isclose(size_min, size_max):
    label_sizes = pd.Series(min(float(FINAL_LABEL_MAX), max(size_min, float(SIZE_FALLBACK))), index=df_cluster_labels.index)
elif size_max < float(FINAL_LABEL_MAX):
    label_sizes = size_min + (label_sizes - size_min) * (float(FINAL_LABEL_MAX) - size_min) / (size_max - size_min)
else:
    label_sizes = label_sizes.clip(upper=float(FINAL_LABEL_MAX))
df_cluster_labels["label_size"] = label_sizes.round(1)

if "apply_label_repel" in globals():
    repel_xy = apply_label_repel(
        df_cluster_labels,
        x_col="x",
        y_col="y",
        size_col="label_size",
        iterations=LABEL_REPEL_ITERS,
        anchor_pull=LABEL_REPEL_ANCHOR,
        step=LABEL_REPEL_STEP,
    )
    df_cluster_labels["x_label"] = repel_xy["x_label"]
    df_cluster_labels["y_label"] = repel_xy["y_label"]
else:
    df_cluster_labels["x_label"] = df_cluster_labels["x"]
    df_cluster_labels["y_label"] = df_cluster_labels["y"]

community_sizes = df_cluster_labels["community"].value_counts().sort_index()
compactness_by_community = {
    int(c_id): _community_compactness(sub)
    for c_id, sub in df_cluster_labels.groupby("community")
}
compactness_vals = np.array(list(compactness_by_community.values()), dtype=float)
comp_min, comp_max = float(compactness_vals.min()), float(compactness_vals.max())

community_styles = {}
for c_id, value in compactness_by_community.items():
    color, fill_alpha = _compactness_to_blob_style(int(c_id))
    community_styles[int(c_id)] = {"color": color, "fill_alpha": fill_alpha}

interpretation_rows = []
for c_id in sorted(community_sizes.index):
    sub_c = df_cluster_labels[df_cluster_labels["community"] == c_id]
    n_members = int(len(sub_c))
    top_country = str(sub_c["country"].value_counts().index[0]) if n_members else "Unknown"
    top_sector = str(sub_c["sector"].value_counts().index[0]) if n_members else "Unknown"
    anchor = str(sub_c.sort_values("ranking_score", ascending=False).iloc[0]["actor"]) if n_members else f"Group {c_id}"
    label_short = f"{top_sector} - {anchor}"
    sample_actors = ", ".join(sub_c["actor"].head(5).tolist())
    interpretation_rows.append(
        {
            "community": int(c_id),
            "label_short": label_short,
            "n_actors": n_members,
            "top_country": top_country,
            "top_sector": top_sector,
            "compactness": float(compactness_by_community[int(c_id)]),
            "sample_actors": sample_actors,
        }
    )

community_annotations = pd.DataFrame(interpretation_rows)
interpretation_path = EXPORTS_DIR / "community_labels_short.csv"
community_annotations.to_csv(interpretation_path, index=False)

communities_path = EXPORTS_DIR / "communities_kmeans_2d.csv"
df_cluster_labels.to_csv(communities_path, index=False)

country_colors = build_country_colors(sorted(df_cluster_labels["country"].dropna().unique().tolist()))

fig_louvain = go.Figure()
x_range = float(df_cluster_labels["x"].max() - df_cluster_labels["x"].min())
y_range = float(df_cluster_labels["y"].max() - df_cluster_labels["y"].min())

for c_id in sorted(community_sizes.index):
    sub = df_cluster_labels[df_cluster_labels["community"] == c_id].copy()
    points_xy = sub[["x", "y"]].to_numpy(dtype=float)
    poly = _patatoid_blob(points_xy, x_range=x_range, y_range=y_range, seed=RANDOM_SEED + int(c_id) * 101)
    label_name = community_annotations.loc[community_annotations["community"] == c_id, "label_short"].iloc[0]
    style = community_styles[int(c_id)]

    fig_louvain.add_trace(
        go.Scatter(
            x=np.r_[poly[:, 0], poly[0, 0]],
            y=np.r_[poly[:, 1], poly[0, 1]],
            mode="lines",
            line=dict(width=0),
            fill="toself",
            fillcolor=_hex_to_rgba(style["color"], style["fill_alpha"]),
            name=label_name,
            hovertemplate=(f"{label_name}<br>Community: {int(c_id)}<extra></extra>"),
            showlegend=True,
        )
    )

for country in sorted(df_cluster_labels["country"].dropna().unique().tolist()):
    sub = df_cluster_labels[df_cluster_labels["country"] == country].copy().sort_values("label_size", ascending=True)
    if sub.empty:
        continue

    fig_louvain.add_trace(
        go.Scatter(
            x=sub["x_label"],
            y=sub["y_label"],
            mode="text",
            text=sub["actor"],
            textposition="middle center",
            textfont=dict(color=country_colors.get(country, "#374151")),
            customdata=np.stack([sub["country"], sub["sector"], sub["ranking_score"], sub["community"]], axis=-1),
            hovertemplate=(
"<b>%{text}</b><br>"
"Country: %{customdata[0]}<br>"
"Sector: %{customdata[1]}<br>"
"Ranking score: %{customdata[2]:,.0f}<br>"
"Community: %{customdata[3]}<extra></extra>"
),
            showlegend=False,
        )
    )

fig_louvain.update_layout(
    title=f"t-SNE-2D - Communities on 2D map ({community_method})",
    xaxis=dict(title="Dimension 1", showgrid=False, zeroline=False),
    yaxis=dict(title="Dimension 2", showgrid=False, zeroline=False),
    legend=dict(title="Communities", font=dict(size=10)),
    font=dict(family="Inter, sans-serif", size=11),
    plot_bgcolor="#FDFAF4",
    paper_bgcolor="#FDFAF4",
    hovermode="closest",
    width=1400,
    height=1100,
)

fig_louvain_html = EXPORTS_DIR / "competition_map_2d_kmeans.html"
fig_louvain.write_html(str(fig_louvain_html))

png_path = EXPORTS_DIR / "competition_map_2d.png"
try:
    fig_louvain.write_image(str(png_path), width=1400, height=1100, scale=2)
    print(f"PNG export -> {png_path}")
except Exception as e:
    print(f"PNG export skipped ({type(e).__name__}): {e}")

print(f"Groups detected: {len(communities)} ({community_method})")
print("Noise points: 0")
print(f"Blob groups (size >= 2): {int((community_sizes >= 2).sum())}")
print(f"Interpretation CSV -> {interpretation_path}")
print(f"Group export -> {communities_path}")
print(f"Map export -> {fig_louvain_html}")
print("Blob color + opacity: discrete multicolor palette with low opacity")

display(community_annotations)
fig_louvain.show()

print("-- Export summary --")
export_paths = [raw_path, long_path, agg_path, cooc_path, coords_path, fig_html]

for maybe_var in ["communities_path", "fig_louvain_html", "interpretation_path"]:
    if maybe_var in globals():
        export_paths.append(globals()[maybe_var])

for p in [
    EXPORTS_DIR / "communities_kmeans_2d.csv",
    EXPORTS_DIR / "community_labels_short.csv",
    EXPORTS_DIR / "competition_map_2d_kmeans.html",
]:
    if p not in export_paths:
        export_paths.append(p)

seen = set()
ordered_paths = []
for p in export_paths:
    p = Path(p)
    key = str(p)
    if key not in seen:
        seen.add(key)
        ordered_paths.append(p)

for p in ordered_paths:
    if p.exists():
        size_kb = p.stat().st_size / 1024
        print(f"  {p.name:<42} {size_kb:6.1f} KB")
    else:
        print(f"  {p.name:<42} {'MISSING':>6}")

PNG export -> C:\Users\33623\Documents\___Projets\AI\Reseaux d'acteurs\analyses\exports\competition_map_2d.png
Groups detected: 21 (louvain_2d(k_neighbors=3,resolution=3.2,min_refined_size=1))
Noise points: 0
Blob groups (size >= 2): 21
Interpretation CSV -> C:\Users\33623\Documents\___Projets\AI\Reseaux d'acteurs\analyses\exports\community_labels_short.csv
Group export -> C:\Users\33623\Documents\___Projets\AI\Reseaux d'acteurs\analyses\exports\communities_kmeans_2d.csv
Map export -> C:\Users\33623\Documents\___Projets\AI\Reseaux d'acteurs\analyses\exports\competition_map_2d_kmeans.html
Blob color + opacity: discrete multicolor palette with low opacity


,community,label_short,n_actors,top_country,top_sector,compactness,sample_actors
0,0,AI model - OpenAI,8,China,AI model,3.628571,"01.AI, 4Paradigm, DeepSeek, MiniMax, Moonshot AI"
1,1,Cloud Provider - Google,8,United States,Cloud Provider,3.391035,"Alibaba, Alphabet, Amazon, Apple, Google"
2,2,Unknown - Cisco,6,United States,Unknown,3.606729,"Cisco, Cognition AI, ContentSquare, DeepL, GitHub"
3,3,Health & Social Care - Abridge,5,United States,Health & Social Care,2.331765,"Abridge, Ambiance Healthcare, Corti, Doctolib,..."
4,4,Financial Services - Ant Group,4,China,Financial Services,2.472088,"Ant Group, Black Forest Labs, Kaola, Weibo"
5,5,Robotics - Figure AI,4,United States,Robotics,1.564894,"1X Technologies, Boston Dynamics, Figure AI, P..."
6,6,Computer Vision - Cognex,2,United States,Computer Vision,0.690393,"Cognex, Prophesee"
7,7,AI model - Helsing,6,France,AI model,3.019059,"Comand AI, Dynatrace, Harmattan AI, Helsing, P..."
8,8,Hardware - Nvidia,8,United States,Hardware,3.241978,"AMD, ARM Holdings, Broadcom, Intel, Mobileye"
9,9,AI model - LightOn,2,France,AI model,0.735396,"LightOn, Yseop"


-- Export summary --
  competitors_raw.csv                          16.6 KB
  competitors_long.csv                         44.8 KB
  competitors_aggregated.csv                   18.1 KB
  cooccurrence_matrix.csv                     219.1 KB
  coords_2d.csv                                 8.0 KB
  competition_map_2d.html                    4789.7 KB
  communities_kmeans_2d.csv                    14.1 KB
  competition_map_2d_kmeans.html             4836.1 KB
  community_labels_short.csv                    2.5 KB


In [218]:
# Data-grounded community names and expanded discrete palette.
with sqlite3.connect(DB_PATH) as con:
    description_rows = con.execute(
        "SELECT name, COALESCE(description, '') FROM enterprises"
    ).fetchall()

description_by_actor = {str(name): str(description) for name, description in description_rows}

THEME_VOCABULARY = {
    "Web Infrastructure Masters": {
        "cloud", "database", "infrastructure", "enterprise", "network", "web",
        "security", "server", "software", "platform", "distributed", "operations",
    },
    "Frontier Model Foundry": {
        "foundation", "language model", "generative", "multimodal", "training",
        "inference", "reasoning", "model", "llm", "research",
    },
    "AI Compute Titans": {
        "gpu", "chip", "semiconductor", "processor", "accelerator", "compute",
        "hardware", "silicon", "datacenter",
    },
    "Clinical Intelligence": {
        "health", "clinical", "patient", "medical", "diagnostic", "hospital",
        "care", "biotech", "drug", "genomic",
    },
    "Defense Autonomy": {
        "defense", "military", "autonomous", "drone", "security", "mission",
        "surveillance", "aerospace",
    },
    "Robotics Vanguard": {
        "robot", "robotics", "humanoid", "automation", "machine", "physical",
        "autonomous", "manufacturing",
    },
    "Autonomous Mobility": {
        "vehicle", "mobility", "driving", "transport", "automotive", "autonomous",
        "navigation", "fleet",
    },
    "Creative Intelligence Studio": {
        "image", "video", "creative", "media", "design", "music", "content",
        "generation", "visual",
    },
    "Developer Intelligence": {
        "developer", "code", "coding", "software", "observability", "testing",
        "repository", "programming", "deployment",
    },
    "Enterprise Automation": {
        "workflow", "enterprise", "automation", "productivity", "business",
        "operations", "process", "customer", "sales",
    },
    "Data Intelligence Guild": {
        "data", "analytics", "warehouse", "intelligence", "search", "knowledge",
        "database", "insight",
    },
    "Vision Systems": {
        "vision", "camera", "image", "sensor", "visual", "inspection", "recognition",
    },
    "Quantum Frontier": {
        "quantum", "qubit", "computing", "photonic", "processor",
    },
    "Financial Intelligence": {
        "financial", "finance", "banking", "payment", "risk", "fraud", "insurance",
    },
}

SECTOR_THEME_FALLBACK = {
    "ai model": "Frontier Model Foundry",
    "hardware": "AI Compute Titans",
    "cloud provider": "Cloud Compute Builders",
    "robotics": "Robotics Vanguard",
    "computer vision": "Vision Systems",
    "health & social care": "Clinical Intelligence",
    "transport & mobility": "Autonomous Mobility",
    "financial services": "Financial Intelligence",
    "manufacturing & operations": "Enterprise Automation",
    "ict": "Web Infrastructure Masters",
    "professional services": "Applied Intelligence Collective",
}

THEME_VARIANTS = {
    "Web Infrastructure Masters": [
        "Web Infrastructure Masters", "Cloud Infrastructure Guild", "Enterprise Backbone",
        "Data Platform Network", "Distributed Systems Collective", "Web Operations Alliance",
        "Cloud Systems Network",
    ],
    "Frontier Model Foundry": [
        "Frontier Model Foundry", "Language Model Labs", "Generative AI Foundry",
        "Multimodal Research Network", "Model Engineering Collective",
    ],
    "Robotics Vanguard": ["Robotics Vanguard", "Embodied Automation"],
    "Creative Intelligence Studio": ["Creative Intelligence Studio", "Synthetic Media Studio"],
}

GAFAM_ACTORS = {"Google", "Alphabet", "Microsoft", "Amazon", "Apple", "Meta"}
# Terrain-inspired palette (high-contrast cartographic style).
EXTENDED_PALETTE = [
    "#0B4F8A", "#1E6FA8", "#3F8FC1",
    "#A56A2A", "#C08B4A",
    "#7B8F2A", "#5E7F1B", "#466317",
    "#2F4F1F", "#1F3B12",
    "#6F4E37", "#8B6B4A",
    "#B8A089", "#EADCCB",
    "#4D7A6B", "#3F6659", "#7E8F54",
    "#5B6E1D", "#70851F", "#8EA02A",
    "#9BAF5E", "#B9C983", "#D6E2AE",
    "#4E6272", "#627987", "#7E939C",
]
BLOB_FILL_ALPHA = 0.17

def _community_theme(sub: pd.DataFrame) -> tuple[str, str]:
    members = set(sub["actor"].astype(str))
    if len(members & GAFAM_ACTORS) >= 3:
        return "GAFAM", "ecosystem membership"
    descriptions = " ".join(
        description_by_actor.get(actor, "") for actor in sub["actor"].astype(str)
    ).casefold()
    sectors = " ".join(sub["sector"].fillna("").astype(str)).casefold()
    corpus = f"{descriptions} {sectors}"
    scores = {
        theme: sum(corpus.count(term) for term in terms)
        for theme, terms in THEME_VOCABULARY.items()
    }
    best_theme, best_score = max(scores.items(), key=lambda item: item[1])
    if "Oracle" in members and scores["Web Infrastructure Masters"] > 0:
        best_theme = "Web Infrastructure Masters"
        best_score = scores[best_theme]
    if best_score == 0:
        top_sector = str(sub["sector"].value_counts().index[0]).casefold()
        best_theme = SECTOR_THEME_FALLBACK.get(top_sector, "Applied Intelligence Collective")
    evidence = sorted(
        term for term in THEME_VOCABULARY.get(best_theme, set()) if term in corpus
    )
    return best_theme, ", ".join(evidence[:5]) or "sector consensus"

def _distinct_theme_label(base_label: str, occurrence: int) -> str:
    variants = THEME_VARIANTS.get(base_label, [base_label])
    if occurrence < len(variants):
        return variants[occurrence]
    return f"{base_label} {occurrence + 1}"

def _wrap_theme_label(label: str, max_chars: int = 22) -> str:
    lines, current = [], []
    for word in label.split():
        candidate = " ".join([*current, word])
        if current and len(candidate) > max_chars:
            lines.append(" ".join(current))
            current = [word]
        else:
            current.append(word)
    if current:
        lines.append(" ".join(current))
    return "<br>".join(lines)

def _repel_theme_labels(frame: pd.DataFrame, iterations: int = 500) -> pd.DataFrame:
    x = frame["x"].to_numpy(dtype=float)
    y = frame["y"].to_numpy(dtype=float)
    x_min, x_max = float(x.min()), float(x.max())
    y_min, y_max = float(y.min()), float(y.max())
    x_span = max(x_max - x_min, 1.0)
    y_span = max(y_max - y_min, 1.0)
    positions = np.column_stack([(x - x_min) / x_span, (y - y_min) / y_span])
    anchors = positions.copy()

    wrapped = frame["label_short"].map(_wrap_theme_label)
    line_sets = wrapped.str.split("<br>")
    half_widths = line_sets.map(lambda lines: max(map(len, lines)) * 0.0046 + 0.012).to_numpy()
    half_heights = line_sets.map(lambda lines: len(lines) * 0.015 + 0.008).to_numpy()

    for _ in range(iterations):
        displacement = np.zeros_like(positions)
        for left in range(len(positions) - 1):
            for right in range(left + 1, len(positions)):
                delta = positions[right] - positions[left]
                overlap_x = half_widths[left] + half_widths[right] + 0.012 - abs(delta[0])
                overlap_y = half_heights[left] + half_heights[right] + 0.012 - abs(delta[1])
                if overlap_x <= 0 or overlap_y <= 0:
                    continue
                if overlap_x < overlap_y:
                    direction = 1.0 if delta[0] >= 0 else -1.0
                    shift = np.array([direction * overlap_x * 0.52, 0.0])
                else:
                    direction = 1.0 if delta[1] >= 0 else -1.0
                    shift = np.array([0.0, direction * overlap_y * 0.52])
                displacement[left] -= shift
                displacement[right] += shift
        positions += displacement + (anchors - positions) * 0.012
        positions[:, 0] = np.clip(positions[:, 0], 0.02 + half_widths, 0.98 - half_widths)
        positions[:, 1] = np.clip(positions[:, 1], 0.02 + half_heights, 0.98 - half_heights)

    return pd.DataFrame({
        "x_label": positions[:, 0] * x_span + x_min,
        "y_label": positions[:, 1] * y_span + y_min,
    }, index=frame.index)

oracle_rows = df_cluster_labels[df_cluster_labels["actor"] == "Oracle"]
oracle_community = int(oracle_rows.iloc[0]["community"]) if not oracle_rows.empty else None
used_labels = {}
theme_rows = []
for c_id in sorted(community_sizes.index):
    sub_c = df_cluster_labels[df_cluster_labels["community"] == c_id]
    base_label, evidence = _community_theme(sub_c)
    if base_label == "Web Infrastructure Masters" and int(c_id) == oracle_community:
        label_short = "Web Infrastructure Masters"
    else:
        occurrence = used_labels.get(base_label, 0)
        used_labels[base_label] = occurrence + 1
        if base_label == "Web Infrastructure Masters" and oracle_community is not None:
            occurrence += 1
        label_short = _distinct_theme_label(base_label, occurrence)
    theme_rows.append({
        "community": int(c_id),
        "label_short": label_short,
        "n_actors": int(len(sub_c)),
        "top_country": str(sub_c["country"].value_counts().index[0]),
        "top_sector": str(sub_c["sector"].value_counts().index[0]),
        "evidence_terms": evidence,
        "compactness": float(compactness_by_community[int(c_id)]),
        "sample_actors": ", ".join(sub_c["actor"].head(5).tolist()),
    })

community_annotations = pd.DataFrame(theme_rows)
community_ids = sorted(community_sizes.index)
if community_annotations["label_short"].duplicated().any():
    raise AssertionError("Community labels must be unique")
if len(EXTENDED_PALETTE) < len(community_ids):
    raise AssertionError("The discrete palette must cover every community")

actor_names = {actor.casefold() for actor in df_cluster_labels["actor"].astype(str) if len(actor) >= 4}
for label in community_annotations["label_short"].astype(str):
    leaked_names = [
        actor_name for actor_name in actor_names
        if re.search(rf"\b{re.escape(actor_name)}\b", label.casefold())
    ]
    if leaked_names:
        raise AssertionError(f"Company name leaked into community label: {label}")

gafam_groups = [
    int(c_id) for c_id in community_ids
    if len(set(df_cluster_labels.loc[df_cluster_labels["community"] == c_id, "actor"]) & GAFAM_ACTORS) >= 3
]
if gafam_groups:
    gafam_label = community_annotations.loc[
        community_annotations["community"] == gafam_groups[0], "label_short"
    ].iloc[0]
    if gafam_label != "GAFAM":
        raise AssertionError("The GAFAM ecosystem was not labeled as expected")
if oracle_community is not None:
    oracle_label = community_annotations.loc[
        community_annotations["community"] == oracle_community, "label_short"
    ].iloc[0]
    if oracle_label != "Web Infrastructure Masters":
        raise AssertionError("Oracle's community was not labeled as expected")

community_annotations.to_csv(interpretation_path, index=False)
if len(fig_louvain.data) < len(community_ids):
    raise RuntimeError("Unexpected figure trace order: missing community blob traces")
for trace_index, c_id in enumerate(community_ids):
    label_name = community_annotations.loc[
        community_annotations["community"] == c_id, "label_short"
    ].iloc[0]
    color = EXTENDED_PALETTE[trace_index]
    fig_louvain.data[trace_index].fillcolor = _hex_to_rgba(color, BLOB_FILL_ALPHA)
    fig_louvain.data[trace_index].name = label_name
    fig_louvain.data[trace_index].hovertemplate = (
        f"{label_name}<br>Community: {int(c_id)}<extra></extra>"
    )

fig_louvain.data = fig_louvain.data[:len(community_ids)]
label_frame = (
    df_cluster_labels.groupby("community", as_index=False)
    .agg(x=("x", "mean"), y=("y", "mean"))
    .merge(community_annotations, on="community", how="left")
)
label_frame["label_size"] = np.interp(
    label_frame["n_actors"],
    (label_frame["n_actors"].min(), label_frame["n_actors"].max()),
    (11.0, 15.0),
)
repelled_theme_xy = _repel_theme_labels(label_frame)
label_frame[["x_label", "y_label"]] = repelled_theme_xy[["x_label", "y_label"]]

fig_louvain.add_trace(
    go.Scatter(
        x=label_frame["x_label"],
        y=label_frame["y_label"],
        mode="text",
        text=label_frame["label_short"].map(_wrap_theme_label),
        textposition="middle center",
        textfont=dict(
            color="#25302B",
            size=label_frame["label_size"],
            family="Arial Black, sans-serif",
        ),
        customdata=np.stack(
            [label_frame["n_actors"], label_frame["top_sector"], label_frame["community"]],
            axis=-1,
        ),
        hovertemplate=(
            "<b>%{text}</b><br>"
            "Actors: %{customdata[0]}<br>"
            "Dominant sector: %{customdata[1]}<br>"
            "Community: %{customdata[2]}<extra></extra>"
        ),
        showlegend=False,
    )
)

x_padding = (df_cluster_labels["x"].max() - df_cluster_labels["x"].min()) * 0.08
y_padding = (df_cluster_labels["y"].max() - df_cluster_labels["y"].min()) * 0.08
fig_louvain.update_xaxes(range=[
    float(df_cluster_labels["x"].min() - x_padding),
    float(df_cluster_labels["x"].max() + x_padding),
])
fig_louvain.update_yaxes(range=[
    float(df_cluster_labels["y"].min() - y_padding),
    float(df_cluster_labels["y"].max() + y_padding),
])

fig_louvain.write_html(str(fig_louvain_html))
try:
    fig_louvain.write_image(str(png_path), width=1400, height=1100, scale=2)
except Exception as exc:
    print(f"PNG export skipped ({type(exc).__name__}): {exc}")

print(f"Community labels updated: {len(community_annotations)} unique thematic labels")
print("Visible company labels: 0")
print(f"Blob palette: {len(EXTENDED_PALETTE)} discrete colors, alpha={BLOB_FILL_ALPHA:.2f}")
display(community_annotations[["community", "label_short", "n_actors", "evidence_terms"]])
fig_louvain.show()

Community labels updated: 21 unique thematic labels
Visible company labels: 0
Blob palette: 26 discrete colors, alpha=0.17


,community,label_short,n_actors,evidence_terms
0,0,Frontier Model Foundry,8,"generative, language model, llm, model, multim..."
1,1,GAFAM,8,ecosystem membership
2,2,Cloud Infrastructure Guild,6,"cloud, database, enterprise, infrastructure, n..."
3,3,Clinical Intelligence,5,"care, clinical, health, hospital, medical"
4,4,Creative Intelligence Studio,4,"content, creative, design, generation, image"
5,5,Robotics Vanguard,4,"humanoid, machine, manufacturing, physical, robot"
6,6,Vision Systems,2,"camera, image, inspection, sensor, vision"
7,7,Enterprise Backbone,6,"cloud, infrastructure, operations, platform, s..."
8,8,AI Compute Titans,8,"chip, compute, gpu, hardware, processor"
9,9,Language Model Labs,2,"language model, llm, model, multimodal, research"


In [229]:
# Restore company labels, enlarge max company label size, and color by country buckets.
LABEL_TEXT_ALPHA = 0.92
FINAL_COMPANY_LABEL_MAX = 24.0
COMPANY_LABEL_MIN = 7.0
COMPANY_LABEL_FALLBACK = 9.0

REGION_LABEL_COLORS = {
    "China": "#D62828",
    "USA": "#1D4ED8",
    "Europe": "#65A30D",
    "Others": "#475569",
}

def _country_bucket(country_value: str) -> str:
    key = str(country_value).strip().casefold()

    usa_keys = {
        "united states", "united states of america", "usa", "us",
    }
    china_keys = {
        "china", "people's republic of china", "pr china", "prc",
    }
    europe_keys = {
        "albania", "andorra", "austria", "belarus", "belgium", "bosnia and herzegovina",
        "bulgaria", "croatia", "cyprus", "czechia", "czech republic", "denmark", "estonia",
        "finland", "france", "germany", "greece", "hungary", "iceland", "ireland", "italy",
        "latvia", "liechtenstein", "lithuania", "luxembourg", "malta", "moldova", "monaco",
        "montenegro", "netherlands", "north macedonia", "norway", "poland", "portugal", "romania",
        "san marino", "serbia", "slovakia", "slovenia", "spain", "sweden", "switzerland",
        "ukraine", "united kingdom", "vatican city", "kosovo",
    }

    if key in china_keys:
        return "China"
    if key in usa_keys:
        return "USA"
    if key in europe_keys:
        return "Europe"
    return "Others"

community_ids = sorted(community_sizes.index)
community_color_map = {
    int(c_id): EXTENDED_PALETTE[index]
    for index, c_id in enumerate(community_ids)
}

# Keep only the blob traces so rerunning this cell never duplicates labels.
fig_louvain.data = fig_louvain.data[:len(community_ids)]

# Legend is no longer useful for this final view.
for trace in fig_louvain.data:
    trace.showlegend = False
fig_louvain.update_layout(showlegend=False)

raw_sizes = pd.to_numeric(df_cluster_labels["label_size"], errors="coerce").fillna(SIZE_FALLBACK)
if np.isclose(float(raw_sizes.min()), float(raw_sizes.max())):
    actor_label_sizes = np.full(len(raw_sizes), COMPANY_LABEL_FALLBACK)
else:
    actor_label_sizes = np.interp(
        raw_sizes,
        (float(raw_sizes.min()), float(raw_sizes.max())),
        (COMPANY_LABEL_MIN, FINAL_COMPANY_LABEL_MAX),
    )

country_buckets = df_cluster_labels["country"].map(_country_bucket)
actor_label_colors = (
    country_buckets.map(REGION_LABEL_COLORS)
    .fillna(REGION_LABEL_COLORS["Others"])
    .tolist()
)

fig_louvain.add_trace(
    go.Scatter(
        x=df_cluster_labels["x_label"],
        y=df_cluster_labels["y_label"],
        mode="text",
        text=df_cluster_labels["actor"],
        textposition="middle center",
        textfont=dict(
            color=actor_label_colors,
            size=actor_label_sizes,
            family="Inter, sans-serif",
        ),
        customdata=np.stack(
            [
                df_cluster_labels["country"],
                country_buckets,
                df_cluster_labels["sector"],
                df_cluster_labels["ranking_score"],
                df_cluster_labels["community"],
            ],
            axis=-1,
        ),
        hovertemplate=(
            "%{text}<br>"
            "Country: %{customdata[0]}<br>"
            "Region bucket: %{customdata[1]}<br>"
            "Sector: %{customdata[2]}<br>"
            "Ranking score: %{customdata[3]:,.0f}<br>"
            "Community: %{customdata[4]}<extra></extra>"
        ),
        showlegend=False,
        name="Companies",
    )
)

theme_label_colors = [
    _hex_to_rgba(community_color_map[int(c_id)], LABEL_TEXT_ALPHA)
    for c_id in label_frame["community"]
]
theme_label_text = label_frame["label_short"].map(_wrap_theme_label).map(lambda s: f"<i>{s}</i>")
fig_louvain.add_trace(
    go.Scatter(
        x=label_frame["x_label"],
        y=label_frame["y_label"],
        mode="text",
        text=theme_label_text,
        textposition="middle center",
        textfont=dict(
            color=theme_label_colors,
            size=label_frame["label_size"],
            family="Inter, sans-serif",
        ),
        customdata=np.stack(
            [label_frame["label_short"], label_frame["n_actors"], label_frame["top_sector"], label_frame["community"]],
            axis=-1,
        ),
        hovertemplate=(
            "%{customdata[0]}<br>"
            "Actors: %{customdata[1]}<br>"
            "Dominant sector: %{customdata[2]}<br>"
            "Community: %{customdata[3]}<extra></extra>"
        ),
        showlegend=False,
        name="Community themes",
    )
)

fig_louvain.write_html(str(fig_louvain_html))
try:
    fig_louvain.write_image(str(png_path), width=1400, height=1100, scale=2)
except Exception as exc:
    print(f"PNG export skipped ({type(exc).__name__}): {exc}")

print(f"Company labels restored: {len(df_cluster_labels)}")
print(f"Company label size range: {COMPANY_LABEL_MIN:.1f} - {FINAL_COMPANY_LABEL_MAX:.1f}")
print("Company label colors by country bucket: China / USA / Europe / Others")
print("Legend hidden for final community map")
fig_louvain.show()

Company labels restored: 111
Company label size range: 7.0 - 24.0
Company label colors by country bucket: China / USA / Europe / Others
Legend hidden for final community map


In [205]:
# Diagnostic: overlap risk between communities on a fully 2D pipeline.
from itertools import combinations

stats_rows = []
for c_id, sub in df_cluster_labels.groupby("community"):
    pts = sub[["x", "y"]].to_numpy(float)
    cent = pts.mean(axis=0)
    d = np.sqrt(((pts - cent) ** 2).sum(axis=1))
    r50 = float(np.percentile(d, 50)) if len(d) else 0.0
    r90 = float(np.percentile(d, 90)) if len(d) else 0.0
    stats_rows.append({
        "community": int(c_id),
        "n_actors": int(len(sub)),
        "centroid_x": float(cent[0]),
        "centroid_y": float(cent[1]),
        "radius_p50": r50,
        "radius_p90": r90,
    })

comm_stats = pd.DataFrame(stats_rows).sort_values("n_actors", ascending=False).reset_index(drop=True)
display(comm_stats.head(10))

pair_rows = []
for c1, c2 in combinations(sorted(comm_stats["community"].tolist()), 2):
    s1 = df_cluster_labels[df_cluster_labels["community"] == c1]
    s2 = df_cluster_labels[df_cluster_labels["community"] == c2]
    p1 = s1[["x", "y"]].to_numpy(float)
    p2 = s2[["x", "y"]].to_numpy(float)

    d2 = np.sqrt(((p1[:, None, :] - p2[None, :, :]) ** 2).sum(axis=2))
    min_d = float(d2.min())

    c1_xy = comm_stats.loc[comm_stats["community"] == c1, ["centroid_x", "centroid_y"]].iloc[0].to_numpy(float)
    c2_xy = comm_stats.loc[comm_stats["community"] == c2, ["centroid_x", "centroid_y"]].iloc[0].to_numpy(float)
    cent_d = float(np.linalg.norm(c1_xy - c2_xy))

    r1 = float(comm_stats.loc[comm_stats["community"] == c1, "radius_p90"].iloc[0])
    r2 = float(comm_stats.loc[comm_stats["community"] == c2, "radius_p90"].iloc[0])
    overlap_index = (r1 + r2) - cent_d

    pair_rows.append({
        "c1": int(c1),
        "c2": int(c2),
        "min_point_dist_2d": min_d,
        "centroid_dist_2d": cent_d,
        "radius_sum_p90": float(r1 + r2),
        "overlap_index_p90": float(overlap_index),
    })

diag_2d = pd.DataFrame(pair_rows).sort_values(["overlap_index_p90", "min_point_dist_2d"], ascending=[False, True]).reset_index(drop=True)
print("Pairs with highest overlap risk in 2D:")
display(diag_2d.head(12))

likely = diag_2d[diag_2d["overlap_index_p90"] > 0]
print(f"Likely overlapping pairs (p90 envelope criterion): {len(likely)}")
if not likely.empty:
    display(likely.head(12))

,community,n_actors,centroid_x,centroid_y,radius_p50,radius_p90
0,18,9,-9.098767,-17.061222,3.066599,7.184725
1,1,8,-0.818773,3.184805,3.351748,5.273533
2,0,8,14.421472,3.210468,3.355486,4.868106
3,8,8,-17.029836,1.272232,3.098430,5.005803
4,13,7,3.034113,-22.387759,3.092932,4.498462
5,20,7,-27.284688,1.406466,3.861033,4.395216
6,19,6,6.830425,19.010386,2.746274,4.557884
7,2,6,-3.340808,-7.392118,3.888939,4.651831
8,7,6,-14.278132,22.780306,3.021376,4.834709
9,10,6,20.045103,12.108715,3.176417,4.922192


Pairs with highest overlap risk in 2D:


,c1,c2,min_point_dist_2d,centroid_dist_2d,radius_sum_p90,overlap_index_p90
0,10,15,2.268262,5.755305,8.493854,2.738549
1,3,18,4.730018,8.393260,10.542306,2.149046
2,17,18,4.235384,8.784808,10.065730,1.280922
3,2,18,4.609151,11.253696,11.836556,0.582860
4,13,17,3.550846,7.240619,7.379466,0.138847
5,10,12,7.053571,10.342903,10.211507,-0.131396
6,0,10,3.840606,10.526349,9.790298,-0.736051
7,16,19,3.791231,7.256645,6.443103,-0.813542
8,8,20,3.238631,10.255731,9.401019,-0.854712
9,1,2,2.912491,10.873452,9.925364,-0.948089


Likely overlapping pairs (p90 envelope criterion): 5


,c1,c2,min_point_dist_2d,centroid_dist_2d,radius_sum_p90,overlap_index_p90
0,10,15,2.268262,5.755305,8.493854,2.738549
1,3,18,4.730018,8.393260,10.542306,2.149046
2,17,18,4.235384,8.784808,10.065730,1.280922
3,2,18,4.609151,11.253696,11.836556,0.582860
4,13,17,3.550846,7.240619,7.379466,0.138847


In [ ]:
# Experiment: Voronoi regions instead of community blobs.
import plotly.graph_objects as go
from scipy.spatial import Voronoi


def _voronoi_finite_polygons_2d(vor: Voronoi, radius: float | None = None):
    if vor.points.shape[1] != 2:
        raise ValueError("Voronoi requires 2D points")

    new_regions = []
    new_vertices = vor.vertices.tolist()

    center = vor.points.mean(axis=0)
    if radius is None:
        radius = float(np.ptp(vor.points, axis=0).max()) * 4.0

    all_ridges: dict[int, list[tuple[int, int, int]]] = {}
    for (p1, p2), (v1, v2) in zip(vor.ridge_points, vor.ridge_vertices):
        all_ridges.setdefault(p1, []).append((p2, v1, v2))
        all_ridges.setdefault(p2, []).append((p1, v1, v2))

    for p1, region_index in enumerate(vor.point_region):
        vertices = vor.regions[region_index]
        if all(v >= 0 for v in vertices):
            new_regions.append(vertices)
            continue

        ridges = all_ridges[p1]
        new_region = [v for v in vertices if v >= 0]

        for p2, v1, v2 in ridges:
            if v1 < 0 or v2 < 0:
                v = v1 if v1 >= 0 else v2
                tangent = vor.points[p2] - vor.points[p1]
                tangent /= np.linalg.norm(tangent)
                normal = np.array([-tangent[1], tangent[0]])

                midpoint = vor.points[[p1, p2]].mean(axis=0)
                direction = np.sign(np.dot(midpoint - center, normal)) * normal
                far_point = vor.vertices[v] + direction * radius

                new_vertices.append(far_point.tolist())
                new_region.append(len(new_vertices) - 1)

        region_coords = np.asarray([new_vertices[v] for v in new_region])
        c = region_coords.mean(axis=0)
        angles = np.arctan2(region_coords[:, 1] - c[1], region_coords[:, 0] - c[0])
        new_region = [v for _, v in sorted(zip(angles, new_region))]
        new_regions.append(new_region)

    return new_regions, np.asarray(new_vertices)


def _bucket_for_voronoi(country_value: str) -> str:
    if "_country_bucket" in globals():
        return _country_bucket(country_value)

    key = str(country_value).strip().casefold()
    usa_keys = {"united states", "united states of america", "usa", "us"}
    china_keys = {"china", "people's republic of china", "pr china", "prc"}
    europe_keys = {
        "albania", "andorra", "austria", "belarus", "belgium", "bosnia and herzegovina",
        "bulgaria", "croatia", "cyprus", "czechia", "czech republic", "denmark", "estonia",
        "finland", "france", "germany", "greece", "hungary", "iceland", "ireland", "italy",
        "latvia", "liechtenstein", "lithuania", "luxembourg", "malta", "moldova", "monaco",
        "montenegro", "netherlands", "north macedonia", "norway", "poland", "portugal", "romania",
        "san marino", "serbia", "slovakia", "slovenia", "spain", "sweden", "switzerland",
        "ukraine", "united kingdom", "vatican city", "kosovo",
    }
    if key in china_keys:
        return "China"
    if key in usa_keys:
        return "USA"
    if key in europe_keys:
        return "Europe"
    return "Others"


community_centers = (
    df_cluster_labels.groupby("community", as_index=False)
    .agg(x=("x", "mean"), y=("y", "mean"))
    .merge(community_annotations[["community", "label_short"]], on="community", how="left")
)

if len(community_centers) < 3:
    raise RuntimeError("Voronoi experiment needs at least 3 communities")

points = community_centers[["x", "y"]].to_numpy(dtype=float)
vor = Voronoi(points)
regions, vertices = _voronoi_finite_polygons_2d(vor)

x_min = float(df_cluster_labels["x"].min())
x_max = float(df_cluster_labels["x"].max())
y_min = float(df_cluster_labels["y"].min())
y_max = float(df_cluster_labels["y"].max())
x_pad = max((x_max - x_min) * 0.08, 1e-6)
y_pad = max((y_max - y_min) * 0.08, 1e-6)
x0, x1 = x_min - x_pad, x_max + x_pad
y0, y1 = y_min - y_pad, y_max + y_pad

fig_voronoi = go.Figure()
community_order = sorted(community_centers["community"].tolist())
color_map = {int(c_id): EXTENDED_PALETTE[idx % len(EXTENDED_PALETTE)] for idx, c_id in enumerate(community_order)}

for row_idx, row in community_centers.reset_index(drop=True).iterrows():
    region = regions[row_idx]
    polygon = vertices[region].copy()
    polygon[:, 0] = np.clip(polygon[:, 0], x0, x1)
    polygon[:, 1] = np.clip(polygon[:, 1], y0, y1)

    c_id = int(row["community"])
    label_short = str(row["label_short"] if pd.notna(row["label_short"]) else f"Community {c_id}")
    fig_voronoi.add_trace(
        go.Scatter(
            x=np.r_[polygon[:, 0], polygon[0, 0]],
            y=np.r_[polygon[:, 1], polygon[0, 1]],
            mode="lines",
            line=dict(width=0),
            fill="toself",
            fillcolor=_hex_to_rgba(color_map[c_id], 0.14),
            hovertemplate=f"{label_short}<br>Community: {c_id}<extra></extra>",
            showlegend=False,
            name=label_short,
        )
    )

raw_sizes = pd.to_numeric(df_cluster_labels["label_size"], errors="coerce").fillna(float(SIZE_FALLBACK))
label_min = 7.0
label_max = float(FINAL_COMPANY_LABEL_MAX if "FINAL_COMPANY_LABEL_MAX" in globals() else 14.0)
label_fallback = 9.0
if np.isclose(float(raw_sizes.min()), float(raw_sizes.max())):
    actor_label_sizes = np.full(len(raw_sizes), label_fallback)
else:
    actor_label_sizes = np.interp(raw_sizes, (float(raw_sizes.min()), float(raw_sizes.max())), (label_min, label_max))

if "REGION_LABEL_COLORS" in globals():
    region_colors = REGION_LABEL_COLORS
else:
    region_colors = {"China": "#D62828", "USA": "#1D4ED8", "Europe": "#65A30D", "Others": "#475569"}

country_buckets = df_cluster_labels["country"].map(_bucket_for_voronoi)
actor_label_colors = country_buckets.map(region_colors).fillna(region_colors["Others"]).tolist()

fig_voronoi.add_trace(
    go.Scatter(
        x=df_cluster_labels["x_label"],
        y=df_cluster_labels["y_label"],
        mode="text",
        text=df_cluster_labels["actor"],
        textposition="middle center",
        textfont=dict(color=actor_label_colors, size=actor_label_sizes, family="Inter, sans-serif"),
        customdata=np.stack(
            [
                df_cluster_labels["country"],
                country_buckets,
                df_cluster_labels["sector"],
                df_cluster_labels["ranking_score"],
                df_cluster_labels["community"],
            ],
            axis=-1,
        ),
        hovertemplate=(
            "%{text}<br>"
            "Country: %{customdata[0]}<br>"
            "Region bucket: %{customdata[1]}<br>"
            "Sector: %{customdata[2]}<br>"
            "Ranking score: %{customdata[3]:,.0f}<br>"
            "Community: %{customdata[4]}<extra></extra>"
        ),
        showlegend=False,
        name="Companies",
    )
)

if "label_frame" in globals() and all(c in label_frame.columns for c in ["x_label", "y_label", "label_short", "label_size"]):
    theme_df = label_frame.copy()
else:
    theme_df = community_centers.copy()
    theme_df["x_label"] = theme_df["x"]
    theme_df["y_label"] = theme_df["y"]
    theme_df["label_size"] = 12.0

theme_colors = [
    _hex_to_rgba(color_map[int(c_id)], 0.92)
    for c_id in theme_df["community"]
]

theme_text = (
    theme_df["label_short"].map(_wrap_theme_label)
    .map(lambda s: f"<i>{s}</i>")
    if "_wrap_theme_label" in globals()
    else theme_df["label_short"].astype(str).map(lambda s: f"<i>{s}</i>")
)

fig_voronoi.add_trace(
    go.Scatter(
        x=theme_df["x_label"],
        y=theme_df["y_label"],
        mode="text",
        text=theme_text,
        textposition="middle center",
        textfont=dict(color=theme_colors, size=theme_df["label_size"], family="Inter, sans-serif"),
        customdata=np.stack([theme_df["label_short"], theme_df["community"]], axis=-1),
        hovertemplate="%{customdata[0]}<br>Community: %{customdata[1]}<extra></extra>",
        showlegend=False,
        name="Community themes",
    )
)

fig_voronoi.update_layout(
    title="t-SNE-2D - Communities with Voronoi regions (experiment)",
    xaxis=dict(title="Dimension 1", showgrid=False, zeroline=False, range=[x0, x1]),
    yaxis=dict(title="Dimension 2", showgrid=False, zeroline=False, range=[y0, y1]),
    font=dict(family="Inter, sans-serif", size=11),
    plot_bgcolor="#FDFAF4",
    paper_bgcolor="#FDFAF4",
    hovermode="closest",
    showlegend=False,
    width=1400,
    height=1100,
)

voronoi_html_path = EXPORTS_DIR / "competition_map_2d_voronoi.html"
voronoi_png_path = EXPORTS_DIR / "competition_map_2d_voronoi.png"
fig_voronoi.write_html(str(voronoi_html_path))
try:
    fig_voronoi.write_image(str(voronoi_png_path), width=1400, height=1100, scale=2)
except Exception as exc:
    print(f"PNG export skipped ({type(exc).__name__}): {exc}")

print(f"Voronoi map export -> {voronoi_html_path}")
print(f"Company labels: {len(df_cluster_labels)}")
print("Region colors: China / USA / Europe / Others")
fig_voronoi.show()